In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-21")
v_file_date = dbutils.widgets.get("p_file_date")

###### Read all the required data

In [0]:
drivers_df = spark.read.parquet(f"{processed_path}/drivers") \
.withColumnRenamed("number", "driver_number") \
.withColumnRenamed("name", "driver_name") \
.withColumnRenamed("nationality", "driver_nationality") 

In [0]:
constructors_df = spark.read.parquet(f"{processed_path}/constructors") \
.withColumnRenamed("name", "team") 

In [0]:
circuits_df = spark.read.parquet(f"{processed_path}/circuits") \
.withColumnRenamed("location", "circuit_location") 

In [0]:
races_df = spark.read.parquet(f"{processed_path}/races") \
.withColumnRenamed("name", "race_name") 

In [0]:
results_df = spark.read.parquet(f"{processed_path}/results") \
.filter(f"file_date = '{v_file_date}'") \
.withColumnRenamed("time", "race_time") \
.withColumnRenamed("race_id", "result_race_id") \
.withColumnRenamed("file_date", "result_file_date") 

###### Join races to circuits

In [0]:
race_circuits_df = races_df.join(circuits_df, races_df.circuit_id == circuits_df.circuit_id, "inner") \
.select(races_df.race_id, races_df.race_year, races_df.race_name, circuits_df.circuit_location)

###### Join results to other dataframes

In [0]:
race_results_df = results_df.join(race_circuits_df, results_df.result_race_id == race_circuits_df.race_id) \
                            .join(drivers_df, results_df.driver_id == drivers_df.driver_id) \
                            .join(constructors_df, results_df.constructor_id == constructors_df.constructor_id)

In [0]:
final_df = race_results_df.select("race_id", "race_year", "race_name", "circuit_location", "driver_name", "driver_number", "driver_nationality",
                                 "team", "grid", "fastest_lap", "race_time", "points", "position", "result_file_date") \
                          .withColumn("created_date", current_timestamp()) \
                          .withColumnRenamed("result_file_date", "file_date")

In [0]:
# display(final_df.filter("race_year == 2020 and race_name == 'Abu Dhabi Grand Prix'").orderBy(final_df.points.desc()))

In [0]:
# final_df.write.mode("overwrite").parquet(f"{presentation_path}/race_results")

In [0]:
overwrite_partition_free(final_df, 'f1_presentation', 'race_results', 'race_id')

In [0]:
# %sql
# drop table if exists f1_presentation.race_results